In [1]:
from src.utils.notebook_setup import *
import src.utils.notebook_ploting as nb_plot
import src.utils.features as features
setup_pandas()

In [2]:
df = load_dataset()

In [3]:
numerical_features, categorical_features = features.split_features(df)
features_df = df[numerical_features].copy()

In [4]:
from src.configs.paths import ARTIFACTS_DIR

path = ARTIFACTS_DIR / "high_correlation_pairs.parquet"
high_corr_pairs = load_parquet(path)

In [5]:
high_corr_pairs

,feature_1,feature_2,correlation
2096,bidirectional_urg_packets,src2dst_urg_packets,1.00
94,bidirectional_packets,bidirectional_ack_packets,1.00
452,dst2src_packets,dst2src_ack_packets,1.00
273,src2dst_packets,src2dst_ack_packets,1.00
3,bidirectional_duration_ms,src2dst_duration_ms,1.00
1980,bidirectional_cwr_packets,src2dst_cwr_packets,1.00
59,bidirectional_packets,bidirectional_bytes,1.00
151,bidirectional_bytes,bidirectional_ack_packets,1.00
1396,bidirectional_max_piat_ms,src2dst_max_piat_ms,0.99
2154,bidirectional_ack_packets,src2dst_ack_packets,0.99


In [6]:
# verificar se são valores iguais para todos os registros

perfect_pairs = high_corr_pairs[high_corr_pairs['correlation'] > 0.99 ][['feature_1', 'feature_2']].values.tolist()

for pair in perfect_pairs:
    result = np.allclose(df[pair[0]], df[pair[1]])
    print(f"{pair[0]} , {pair[1]}: {result}")



bidirectional_urg_packets , src2dst_urg_packets: True
bidirectional_packets , bidirectional_ack_packets: False
dst2src_packets , dst2src_ack_packets: False
src2dst_packets , src2dst_ack_packets: False
bidirectional_duration_ms , src2dst_duration_ms: False
bidirectional_cwr_packets , src2dst_cwr_packets: False
bidirectional_packets , bidirectional_bytes: False
bidirectional_bytes , bidirectional_ack_packets: False
bidirectional_max_piat_ms , src2dst_max_piat_ms: False
bidirectional_ack_packets , src2dst_ack_packets: False
bidirectional_packets , src2dst_packets: False
bidirectional_packets , src2dst_ack_packets: False
src2dst_packets , bidirectional_ack_packets: False
bidirectional_stddev_ps , bidirectional_max_ps: False


In [7]:
direction_pairs = [
    ("bidirectional_urg_packets", "src2dst_urg_packets"),

    ("bidirectional_packets", "src2dst_packets"),
    ("bidirectional_packets", "dst2src_packets"),
    ("src2dst_packets", "dst2src_packets"),

    ("bidirectional_duration_ms", "src2dst_duration_ms"),
    ("bidirectional_duration_ms", "dst2src_duration_ms"),
    ("src2dst_duration_ms", "dst2src_duration_ms"),

    ("bidirectional_ack_packets", "src2dst_ack_packets"),
    ("bidirectional_ack_packets", "dst2src_ack_packets"),
    ("src2dst_ack_packets", "dst2src_ack_packets"),

    ("bidirectional_cwr_packets", "src2dst_cwr_packets"),
    ("bidirectional_ece_packets", "dst2src_ece_packets"),
    ("bidirectional_syn_packets", "src2dst_syn_packets"),
    ("bidirectional_fin_packets", "src2dst_fin_packets"),
    ("bidirectional_fin_packets", "dst2src_fin_packets"),
]

tcp_flag_pairs = [
    ("bidirectional_packets", "bidirectional_ack_packets"),
    ("dst2src_packets", "dst2src_ack_packets"),
    ("src2dst_packets", "src2dst_ack_packets"),
    ("bidirectional_bytes", "bidirectional_ack_packets"),
    ("bidirectional_packets", "src2dst_ack_packets"),
    ("bidirectional_packets", "dst2src_ack_packets"),
    ("dst2src_packets", "bidirectional_ack_packets"),
    ("src2dst_packets", "bidirectional_ack_packets"),
    ("bidirectional_bytes", "dst2src_ack_packets"),
    ("bidirectional_bytes", "src2dst_ack_packets"),
]

volume_pairs = [
    ("bidirectional_packets", "bidirectional_bytes"),
    ("bidirectional_bytes", "dst2src_packets"),
    ("bidirectional_bytes", "src2dst_packets"),
]

ps_pairs = [
    ("bidirectional_stddev_ps", "bidirectional_max_ps"),
    ("dst2src_stddev_ps", "dst2src_max_ps"),
    ("bidirectional_max_ps", "dst2src_max_ps"),
    ("bidirectional_stddev_ps", "dst2src_max_ps"),
    ("bidirectional_stddev_ps", "dst2src_stddev_ps"),
    ("bidirectional_max_ps", "dst2src_stddev_ps"),
    ("bidirectional_mean_ps", "dst2src_mean_ps"),
    ("bidirectional_stddev_ps", "dst2src_mean_ps"),
    ("dst2src_mean_ps", "dst2src_max_ps"),
    ("src2dst_stddev_ps", "src2dst_max_ps"),
    ("dst2src_mean_ps", "dst2src_stddev_ps"),
    ("bidirectional_mean_ps", "bidirectional_stddev_ps"),
    ("bidirectional_mean_ps", "bidirectional_max_ps"),
    ("src2dst_mean_ps", "src2dst_stddev_ps"),
    ("bidirectional_mean_ps", "dst2src_max_ps"),
]

piat_pairs = [
    ("bidirectional_max_piat_ms", "src2dst_max_piat_ms"),
    ("bidirectional_mean_piat_ms", "src2dst_mean_piat_ms"),
    ("bidirectional_max_piat_ms", "dst2src_max_piat_ms"),
    ("src2dst_max_piat_ms", "dst2src_max_piat_ms"),
    ("bidirectional_mean_piat_ms", "bidirectional_stddev_piat_ms"),
    ("bidirectional_stddev_piat_ms", "bidirectional_max_piat_ms"),
    ("bidirectional_stddev_piat_ms", "src2dst_max_piat_ms"),
    ("bidirectional_stddev_piat_ms", "src2dst_mean_piat_ms"),
]



In [8]:
columns_to_ignore = features.get_columns_to_ignore()
not_included = set()
for pair in piat_pairs:
    for item in pair:
        if item not in columns_to_ignore:
            not_included.add(item)
print(not_included)


{'bidirectional_mean_piat_ms', 'bidirectional_max_piat_ms', 'bidirectional_stddev_piat_ms'}


In [9]:
columns_to_ignore = features.get_columns_to_ignore()
remaining_columns = [col for col in df.columns if col not in columns_to_ignore]
display(remaining_columns)

['src_port',
 'dst_port',
 'protocol',
 'ip_version',
 'bidirectional_duration_ms',
 'bidirectional_packets',
 'bidirectional_bytes',
 'src2dst_bytes',
 'dst2src_bytes',
 'bidirectional_mean_ps',
 'bidirectional_stddev_ps',
 'bidirectional_max_ps',
 'bidirectional_min_piat_ms',
 'bidirectional_mean_piat_ms',
 'bidirectional_stddev_piat_ms',
 'bidirectional_max_piat_ms',
 'bidirectional_syn_packets',
 'bidirectional_urg_packets',
 'bidirectional_psh_packets',
 'bidirectional_rst_packets',
 'bidirectional_fin_packets',
 'label']

In [10]:
(df["bidirectional_ack_packets"] == 0).mean()

np.float64(0.3275376947080193)